# 🏥 Sistema de Gestión Hospitalaria (SGH)
### Curso de Programación Orientada a Objetos
**Ministerio de Salud y Protección Social — Colombia**

Presentado por: **MOISES DAVID BAQUERO DAZA y KEYNER STEVEN GARCIA ANAYA**

---



## CELDA 0 — Configuración de Google Drive y Rutas

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
RUTA_BASE = '/content/drive/MyDrive/SGH/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_PACIENTES      = RUTA_BASE + 'pacientes.txt'
RUTA_MEDICOS        = RUTA_BASE + 'medicos.txt'
RUTA_ESPECIALIDADES = RUTA_BASE + 'especialidades.txt'
RUTA_CITAS          = RUTA_BASE + 'citas.txt'
RUTA_CONSULTAS      = RUTA_BASE + 'consultas.txt'
RUTA_TRATAMIENTOS   = RUTA_BASE + 'tratamientos.txt'
RUTA_MEDICAMENTOS   = RUTA_BASE + 'medicamentos.txt'

print(' Google Drive montado correctamente.')
print(f' Ruta base del proyecto: {RUTA_BASE}')

## SECCIÓN 1 — Modelo (Dominio)
### 1.1 — Enumeraciones del sistema

In [ ]:
from enum import Enum

class RegimenEnum(Enum):
    """Enum que representa los regímenes de aseguramiento en salud en Colombia."""
    CONTRIBUTIVO = 'CONTRIBUTIVO'
    SUBSIDIADO   = 'SUBSIDIADO'
    ESPECIAL     = 'ESPECIAL'
    VINCULADO    = 'VINCULADO'

class EstadoCitaEnum(Enum):
    """Enum que representa los posibles estados de una cita médica."""
    PROGRAMADA  = 'PROGRAMADA'
    ATENDIDA    = 'ATENDIDA'
    CANCELADA   = 'CANCELADA'
    NO_ASISTIO  = 'NO_ASISTIO'

print('Enumeraciones cargadas: RegimenEnum, EstadoCitaEnum')

### 1.2 — Clase Especialidad

In [ ]:
class Especialidad:
    """Representa una especialidad médica disponible en el hospital."""

    def __init__(self, codigo: str, nombre: str, descripcion: str):
        """
        Inicializa una especialidad médica.

        Args:
            codigo (str): Código único de la especialidad (ej: ESP-001).
            nombre (str): Nombre de la especialidad (ej: Cardiología).
            descripcion (str): Descripción breve de la especialidad.
        """
        self.codigo      = codigo
        self.nombre      = nombre
        self.descripcion = descripcion

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return f'{self.codigo}|{self.nombre}|{self.descripcion}'

    @staticmethod
    def from_linea(linea: str) -> 'Especialidad':
        """Crea un objeto Especialidad a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Especialidad(campos[0], campos[1], campos[2])

    def __str__(self) -> str:
        return f'[{self.codigo}] {self.nombre} — {self.descripcion}'

print('Clase Especialidad cargada.')

### 1.3 — Clase Paciente

In [ ]:
class Paciente:
    """Representa un paciente registrado en el sistema hospitalario."""

    def __init__(self, num_documento: str, nombre: str, fecha_nacimiento: str,
                 tipo_sangre: str, eps: str, regimen: str, antecedentes: str):
        """
        Inicializa un paciente con sus datos personales y de aseguramiento.

        Args:
            num_documento (str): Número de documento de identidad del paciente.
            nombre (str): Nombre completo del paciente.
            fecha_nacimiento (str): Fecha de nacimiento en formato YYYY-MM-DD.
            tipo_sangre (str): Tipo de sangre (ej: O+, A-, B+).
            eps (str): Nombre de la EPS a la que pertenece.
            regimen (str): Régimen de afiliación (CONTRIBUTIVO, SUBSIDIADO, etc.).
            antecedentes (str): Antecedentes médicos relevantes del paciente.
        """
        self.num_documento    = num_documento
        self.nombre           = nombre
        self.fecha_nacimiento = fecha_nacimiento
        self.tipo_sangre      = tipo_sangre
        self.eps              = eps
        self.regimen          = regimen
        self.antecedentes     = antecedentes

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return (f'{self.num_documento}|{self.nombre}|{self.fecha_nacimiento}|'
                f'{self.tipo_sangre}|{self.eps}|{self.regimen}|{self.antecedentes}')

    @staticmethod
    def from_linea(linea: str) -> 'Paciente':
        """Crea un objeto Paciente a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Paciente(campos[0], campos[1], campos[2],
                        campos[3], campos[4], campos[5], campos[6])

    def __str__(self) -> str:
        return (f'Documento: {self.num_documento} | Nombre: {self.nombre} | '
                f'Nacimiento: {self.fecha_nacimiento} | Sangre: {self.tipo_sangre} | '
                f'EPS: {self.eps} | Régimen: {self.regimen} | '
                f'Antecedentes: {self.antecedentes}')

print('Clase Paciente cargada.')

### 1.4 — Clase Medico

In [ ]:
class Medico:
    """Representa un médico registrado en el sistema hospitalario."""

    def __init__(self, num_registro: str, nombre: str, especialidad: str,
                 consultorio: str, horario: str):
        """
        Inicializa un médico con sus datos profesionales.

        Args:
            num_registro (str): Número de registro médico único (ej: MED-001).
            nombre (str): Nombre completo del médico.
            especialidad (str): Código de la especialidad que ejerce.
            consultorio (str): Número o identificador del consultorio asignado.
            horario (str): Franja horaria de atención (ej: 07:00-13:00).
        """
        self.num_registro  = num_registro
        self.nombre        = nombre
        self.especialidad  = especialidad
        self.consultorio   = consultorio
        self.horario       = horario

    def to_linea(self) -> str:
        """Convierte el objeto a una línea de texto con campos separados por pipe."""
        return (f'{self.num_registro}|{self.nombre}|{self.especialidad}|'
                f'{self.consultorio}|{self.horario}')

    @staticmethod
    def from_linea(linea: str) -> 'Medico':
        """Crea un objeto Medico a partir de una línea de texto del archivo."""
        campos = linea.strip().split('|')
        return Medico(campos[0], campos[1], campos[2], campos[3], campos[4])

    def __str__(self) -> str:
        return (f'Registro: {self.num_registro} | Nombre: {self.nombre} | '
                f'Especialidad: {self.especialidad} | Consultorio: {self.consultorio} | '
                f'Horario: {self.horario}')

print('Clase Medico cargada.')